# Women's Cricket World Cup 2025 Outcome Predition

## Introduction

Our goal is to use simple baseline models to predict match outcomes for the recently concluded Women's Cricket World Cup 2025 using historical match data.

Raw ball-by-ball data is sourced from Cricsheet and transformed into match-level feature dataset, each row representing a match.

## Problem Statement

Given match-level features derived from ball-by-ball data, can we predict the outcome of a Women’s Cricket World Cup match?

This is formulated as a supervised learning problem, where historical matches are used to train a model that will be evaluated on matches from the Women’s Cricket World Cup 2025.

## Data Source and Handling

### Raw Data

- **Source:** [Cricsheet](https://cricsheet.org/)
- **Format:** JSON (one file per match)
- **Type:** Ball-by-ball
- **Count:** 159 (some will be ignored, see below for explanation)
- **Scope:** 
	- Only the matches between the eight teams that participated in the Women's Cricket World Cup 2025 will be considered.
	- Only matches after World Cup 2022 are considered.

To keep the repository lightweight, raw data files are not included in the repository. You can obtain it with the following steps.

### Obtaining the Raw Data

1. Download zipped JSON data from [Cricsheet](https://cricsheet.org/downloads/) for WODIs.

2. Unzip and place the JSON files inside `data/raw`.

## Data Processing

### Base Dataset

The base dataset is a **match-level dataset**, where each row represents a single 
match. Raw ball-by-ball JSON data is parsed and transformed to create the base dataset.

The base dataset contains match information such as team names, venue, date, etc. and
foundational match statistics such as runs scored, wickets taken, deliveries played.
From these, a match-level feature dataset is derived.


#### Base Dataset Schema

**Note:** The indexing 0 and 1 is decided alphabetically. So if the match is between, say India and England, then team_0 would be England and team_1 would be India.

| Column Name      | Description                                                                                                               |
| :--------------- | :------------------------------------------------------------------------------------------------------------------------ |
| match_id         | A unique id for the match derived from file name. For example, a match id 1490443 corresponds to the file '1490443.json'. |
| country          | The country where the match venue is located.                                                                             |
| start_date       | Match start date.                                                                                                         |
| team_0           | The first team playing the match (teams ordered alphabetically).                                                          |
| team_1           | The second team playing the match.                                                                                        |
| toss_winner      | Toss winner - 0 if team_0 wins the toss, otherwise 1.                                                                     |
| toss_decision    | Decision taken by the team that won the toss - 0 if they decide to bat first, otherwise 1.                                |
| runs_0 (1)       | Runs scored by team_0 (1)                                                                                                 |
| wickets_0 (1)    | Wickets lost by team_0 (1)                                                                                                |
| deliveries_0 (1) | Deliveries played by team_0 (1)                                                                                           |
| result           | Result of the game - 0 if team_0 wins and 1 if team_1 wins. Tied matches and matches with no result are not considered.   |


The Base dataset can be created from raw ball-by-ball data running the script `src/womenswc/build_base_dataset.py`or importing `main` from `womenswc.build_base_dataset`.

For ease of use, we'll define our data directory as pathlib path.

In [7]:
from pathlib import Path

DATA_DIR = Path("../data").resolve()
print(DATA_DIR)
print(DATA_DIR.is_dir())

/home/siddhant/Documents/projects/womens-wc/data
True


In [8]:
# Build the Base Dataset
from womenswc.build_base_dataset import main

main()

Base dataset created and saved to /home/siddhant/Documents/projects/womens-wc/data/processed/base_dataset.parquet


Let's check if that did the trick.

In [9]:
import pandas as pd

base_df = pd.read_parquet(DATA_DIR / "processed" / "base_dataset.parquet")
print(base_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   match_id       123 non-null    object        
 1   country        123 non-null    object        
 2   start_date     123 non-null    datetime64[ns]
 3   event          123 non-null    object        
 4   team_0         123 non-null    object        
 5   team_1         123 non-null    object        
 6   toss_winner    123 non-null    float64       
 7   toss_decision  123 non-null    float64       
 8   runs_0         123 non-null    float64       
 9   wickets_0      123 non-null    float64       
 10  deliveries_0   123 non-null    float64       
 11  runs_1         123 non-null    float64       
 12  wickets_1      123 non-null    float64       
 13  deliveries_1   123 non-null    float64       
 14  result         123 non-null    int64         
dtypes: datetime64[ns](1), f

We see that the base dataset was successfully created.

## Features

Now that we have the base dataset, we want to create some domain-specific features that may help us predict match outcomes.

### Home Advantage

We've observed that teams play better at home than away becasue the players get acclimatized to home condtions.

We consider a very simple measure of home advantage, We define it to be 1 if a team is the home tea, 0 if venue is neutral for the two teams, and -1 if a teams is the away team. Later on, we may consider team strengths in various conditions such as turning tracks, fast paced tracks, and flat tracks.

### Weighted Decay

Since we want to model recent form, we need to weight the stats, with weights getting smaller for older matches.

Suppose our matches are indexed $0, 1, 2,...$ and we want to see total weighted runs scored by a team T, before match number $i$.
We define the following sequences, each ordered by match start date.

1. **d :** Days since first match start date.
2. **w :** Weights
3. **m:** Mask for filtering
4. **r :** Runs scored by T in the match. (0 if T didn't play the match.)

For this project, we'll use exponentially decaying weights, though you can use your own weight functions.

$$ w_{i} = 2^{-kd_{i}} $$
where $ k = \frac{1}{half-life} $

The total weighted runs scored by T before match $i$:

$$ wt\_runs_{i} = \sum_{j=0}^{i-1}r_{j} \times 2^{-k(d_{i} - d_{j})}$$

$$ wt\_runs_{i} = \sum_{j=0}^{i-1}r_{j} \times \left( \frac{w_{i}}{w_{j}} \right) $$

Since $i$ is independent of $j$, we can move it outside the sum.

$$ wt\_runs_{i} = w_{i} \times \sum_{j=0}^{i-1}\frac{r_{j}}{w_{j}} $$

If our sequences are pandas Series, then this becomes:

```python
wt_runs = w * ((r/w).cumsum() - r/w)
```

We can similarly calculate weighted stats for: wins, match count, wickets lost, and deliveries played.
But these are only batting side's stats. For the bowling side, we simply switch columns like runs_0 with run_1. This works because
at match-level, 

runs scored by the batting side = runs conceded by the bowling side. 

Similarly for wickets and deliveries.

From here on, we will drop the word weighted. It is implied in the case of Win Percentage, Batting Stength, and Bowling Stength.

### Team Strength

We will use several metrics to model team strength for any team before the start of a match.\

#### Win Percentage

Win Percentage: The name is self-explanatory. It's the percentage of matches won by a team.

Win Percentage = 100 x Win Count / Match Count

#### Batting Strength

We use two competing metrics in ODIs, to model a team's batting strength.

1. **Batting Average:**  Runs Scored / Wickets Lost
2. **Batting Strike-rate:** 100 x Runs Scored / Deliveries Played

#### Bowling Strength

We again use two competing metrics to model a team's bowling strength

1. **Bowling Average:** Runs Conceded / Wickets Taken
2. **Bowling Economy:** 6 * Runs Conceded / Deliveries

Now we have our features list:
1. Team_0
2. Team_1
3. Home Advantage_0
4. Win Percentage_0/1
5. Batting Average_0/1
6. Batting Strike-rate_0/1
7. Bowling Average_0/1
8. Bowling Economy_0/1

You can build features database using the script `src/womenswc/features.py` or import main() from womenswc.features and call it.


In [10]:
from womenswc.features import main

main()

Features dataset created and saved to /home/siddhant/Documents/projects/womens-wc/data/processed/features_dataset.parquet


In [13]:
features_df = pd.read_parquet(
    Path("../data/processed/features_dataset.parquet").resolve()
)
features_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 118 entries, 1 to 122
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   team_0             118 non-null    object 
 1   team_1             118 non-null    object 
 2   home_adv_0         118 non-null    int64  
 3   win_percentage_0   118 non-null    float64
 4   batting_average_0  118 non-null    float64
 5   batting_sr_0       118 non-null    float64
 6   bowling_average_0  118 non-null    float64
 7   bowling_economy_0  118 non-null    float64
 8   home_adv_1         118 non-null    int64  
 9   win_percentage_1   118 non-null    float64
 10  batting_average_1  118 non-null    float64
 11  batting_sr_1       118 non-null    float64
 12  bowling_average_1  118 non-null    float64
 13  bowling_economy_1  118 non-null    float64
dtypes: float64(10), int64(2), object(2)
memory usage: 13.8+ KB


In [14]:
features_df.head()

,team_0,team_1,home_adv_0,win_percentage_0,batting_average_0,batting_sr_0,bowling_average_0,bowling_economy_0,home_adv_1,win_percentage_1,batting_average_1,batting_sr_1,bowling_average_1,bowling_economy_1
index,,,,,,,,,,,,,,
1,Australia,England,1,100.0,22.78,68.33,17.80,3.96,-1,0.0,17.80,65.93,22.78,4.10
2,Australia,England,1,100.0,24.01,65.61,15.34,3.40,-1,0.0,15.34,56.59,24.01,3.94
4,India,New Zealand,-1,0.0,21.30,71.48,27.50,5.71,1,100.0,27.50,95.16,21.30,4.29
5,India,New Zealand,-1,0.0,30.25,80.82,32.27,5.63,1,100.0,32.27,93.83,30.25,4.85
6,India,New Zealand,-1,0.0,29.34,85.23,34.55,5.65,1,100.0,34.55,94.20,29.34,5.11
